# Select rows 
Group by user_id, pick N random users, and retain their rows

In [2]:
import pandas as pd

N = 5
df = pd.read_csv(r'./raw/youtube_watch_log.csv')

selected_users = df['user_id'].drop_duplicates().sample(n=N, random_state=42)
df = df[df['user_id'].isin(selected_users)]

len(df)

51318

# Augment with title and thumbnail

In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm 
from concurrent.futures import ThreadPoolExecutor

with open('api-key.txt') as f:
    api_key = f.readline().strip()

def fetch_title(video_id):
    url = f"https://www.googleapis.com/youtube/v3/videos?part=snippet&id={video_id}&key={api_key}"
    data = requests.get(url).json()
    return data['items'][0]['snippet']['localized']['title'] if data['items'] else None

results = {}
with ThreadPoolExecutor() as ex:
    futures = {ex.submit(fetch_title, vid): vid for vid in df['video_id']}
    for future in tqdm(as_completed(futures), total=len(futures)):
        vid = futures[future]
        try:
            results[vid] = future.result()
        except Exception as e:
            print(f"{vid}: {e}")

df['video_title'] = df['video_id'].map(results)

# Grab thumbnails and save to disk

In [ ]:
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

# df = pd.read_csv(r'./clean_youtube_watch_log.csv')
os.makedirs('thumbnails', exist_ok=True)

def download_thumbnail(video_id):
    url = f"https://img.youtube.com/vi/{video_id}/hqdefault.jpg"
    response = requests.get(url, timeout=5)
    if response.status_code == 200:
        with open(f'thumbnails/{video_id}.jpg', 'wb') as f:
            f.write(response.content)

# Execute in parallel to speed up downloading thumbnails
with ThreadPoolExecutor() as ex:
    futures = [ex.submit(download_thumbnail, vid) for vid in df['video_id']]
    for _ in tqdm(as_completed(futures), total=len(futures)):
        pass

  0%|          | 0/987 [00:00<?, ?it/s]

# Export to CSV

In [ ]:
df.to_csv(r'./clean_youtube_watch_log.csv', index=False)